# Module 2 — Les fondamentaux du machine learning

**CONSEIL :** Sauvegardez une copie de ce notebook dans votre Drive avant de commencer !

**Objectifs d'apprentissage**
- Comprendre ce qu'est une **feature** (et pourquoi il en faut)
- Savoir **nettoyer** un dataset sans pleurer
- Faire un **train/test split** sans tricher
- Diagnostiquer un **overfitting** à l'œil nu
- Choisir une **métrique** qui ne ment pas (accuracy, precision, recall)
- Passer de la classification à la **régression** sans tout réécrire

## Le running example : Law School

Un dataset classique en sociologie quantitative du droit et en fairness ML : la *LSAC National Longitudinal Bar Passage Study* (Wightman, 1998). On suit environ 18 700 étudiant·es de droit aux États-Unis, depuis leur entrée en faculté jusqu'à leur passage du barreau (l'examen national pour exercer comme avocat·e).

Le parcours :

```
LSAT + UGPA → 1ère année → ... → Law School GPA → Bar Exam → Pass/Fail
```

On prédira **`y_pass_bar`** : a-t-on réussi le barreau (0/1) ? À partir des seules variables connues **à l'entrée** en faculté de droit (notes pré-droit, statut temps plein, revenu familial, tier de la faculté).

Le dataset est livré déjà nettoyé et déjà splitté (train/test) dans `data/law_school/`. Voir `data/law_school/codebook.md` pour le détail des colonnes.

## Setup

In [ ]:
# !pip install -q scikit-learn pandas matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Chargement des CSV pré-traités (train + test pour la classification)
URL = "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/data/law_school"

df_train = pd.read_csv(f"{URL}/law_school_for_classification_train.csv")
df_test = pd.read_csv(f"{URL}/law_school_for_classification_test.csv")

print(f"Train : {len(df_train):,} lignes")
print(f"Test  : {len(df_test):,} lignes")
df_train.head()

## 2.1 La notion de feature

Une **feature** (variable explicative, prédicteur…) est une colonne dont le modèle se sert pour faire sa prédiction.

Dans `data/law_school/`, les colonnes suivent une convention :

| Préfixe | Rôle | Colonnes ici |
| :-: | - | - |
| `x_` | features prédictives | `x_lsat`, `x_ugpa`, `x_fulltime`, `x_fam_inc`, `x_tier` |
| `y_` | cible à prédire | `y_pass_bar` |
| `z_` | attribut sensible | `z_white` |

Les `x_*` entrent dans `X`. Le `y_*` est ce qu'on essaye de prédire. Le `z_*` est l'attribut sensible que l'on **isole** : on évite de l'utiliser comme feature directe, et on s'en servira plus tard pour vérifier que le modèle n'est pas injuste (cours de jeudi). Pour ce module, on l'écarte simplement de `X`.

Les modèles **adorent** le numérique, **tolèrent mal** le catégoriel brut. Ici tout est déjà numérique (les catégories ont été encodées en codes ordinaux dans le dataset source), donc on n'a pas à se battre avec du `one-hot encoding`. Veinard·es.

In [ ]:
# Aperçu des types
df_train.dtypes

In [ ]:
# Distribution de la cible
df_train["y_pass_bar"].value_counts(normalize=True).round(3)

### Hack Time

Dans la cellule ci-dessous :
- Affichez `df_train.describe()` pour voir les distributions des features numériques
- Repérez la feature qui a la plus grande dispersion (écart-type / range)
- Comparez la proportion de `z_white == 1` dans le train et dans le test. Est-elle bien préservée par le split ?
- Réfléchissez : dans **votre** domaine de recherche, quelles features auraient un rôle analogue à `x_lsat` et `x_ugpa` (signaux disponibles **avant** l'événement à prédire) ?

In [ ]:
# Votre code ici

## 2.2 Nettoyage et préparation

Règle d'or : **garbage in, garbage out**. Un modèle entraîné sur des données pourries fait des prédictions pourries (et a souvent l'air confiant en le faisant).

Trois grands chantiers, en général :
- Les **valeurs manquantes**
- Les **outliers** (vraies erreurs ou vrais individus extrêmes ?)
- La **normalisation** des numériques (certains modèles y sont très sensibles, d'autres s'en fichent)

Bonne nouvelle : le script `data/law_school/fetch_law_school.py` a déjà fait le sale boulot avant qu'on arrive (drop des NA, recodage, retrait des variables qui causeraient du **target leakage** — par exemple les notes obtenues *pendant* les études de droit, qui rendraient la prédiction triviale et inutile). C'est documenté dans le codebook.

On vérifie quand même, parce que faire confiance aveuglément aux données qu'on nous tend est statistiquement déconseillé.

In [ ]:
# Sanity check : reste-t-il des NA ?
df_train.isna().sum()

### Hack Time

- Avec `df_train.describe()`, regardez les valeurs `min` et `max` de `x_lsat` et `x_ugpa`. Y a-t-il des valeurs aberrantes (un GPA undergrad à 12 sur 4, par exemple) ?
- Tracez l'histogramme de `x_lsat` (avec `df_train["x_lsat"].hist()`). La distribution a-t-elle l'air gaussienne ? Bi-modale ?
- Question conceptuelle : pourquoi est-ce qu'on a **exclu** la note de première année (`zfygpa`) du dataset livré, alors qu'elle prédirait excellemment bien le passage du barreau ?

In [ ]:
# Votre code ici

## 2.3 Train / test split

L'objectif d'un modèle ML n'est **pas** de bien marcher sur les données d'entraînement. C'est de **généraliser** à des données non vues.

D'où la nécessité de **séparer** les données en deux : on entraîne le modèle sur 75 % (le « train ») et on mesure sa qualité sur les 25 % restants (le « test »). Le modèle n'a jamais vu ces 25 % pendant l'entraînement, donc s'il s'en sort bien là-dessus, on peut espérer qu'il marchera aussi sur de vraies nouvelles données.

Ici, le split est **déjà fait** dans les fichiers. C'est une pratique courante : la personne qui prépare les données fournit directement `_train.csv` et `_test.csv`, ce qui garantit que personne ne triche en regardant le test pendant l'exploration.

> Quand on doit le faire soi-même, l'outil standard est `sklearn.model_selection.train_test_split(X, y, test_size=0.25, stratify=y)`. La **stratification** sur la cible garantit que le test set a à peu près la même proportion de réussite au barreau que le train set.

In [ ]:
# Préparer X et y depuis les CSV pré-splittés
FEATURES = ["x_lsat", "x_ugpa", "x_fulltime", "x_fam_inc", "x_tier"]
TARGET = "y_pass_bar"

X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_test = df_test[FEATURES]
y_test = df_test[TARGET]

print(f"X_train : {X_train.shape}, y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}, y_test  : {y_test.shape}")
print(f"Taux de réussite train : {y_train.mean():.3f}")
print(f"Taux de réussite test  : {y_test.mean():.3f}")

## 2.4 Overfitting

Le **piège classique du ML** : le modèle apprend ses données par cœur, performe à 100 % sur le train, et plante royalement sur le test.

Symptôme : précision sur le train >> précision sur le test.

L'**arbre de décision** est la machine à fabriquer de l'overfitting : laissé sans contrainte sur sa profondeur, il mémorise littéralement les données d'entraînement.

(Un arbre de décision, c'est une suite de questions en cascade : « LSAT > 35 ? Oui → UGPA > 3.5 ? Non → temps plein ? »… Plus l'arbre est profond, plus il pose de questions, plus il mémorise.)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Squelette : entraîner des arbres de profondeur croissante et observer la divergence
# depths = [1, 3, 5, 10, 20, None]
# train_scores, test_scores = [], []
# for d in depths:
#     clf = DecisionTreeClassifier(max_depth=d, random_state=42)
#     clf.fit(X_train, y_train)
#     train_scores.append(clf.score(X_train, y_train))
#     test_scores.append(clf.score(X_test, y_test))
#
# xs = [d if d is not None else 30 for d in depths]
# plt.plot(xs, train_scores, "o-", label="train")
# plt.plot(xs, test_scores, "o-", label="test")
# plt.xlabel("Profondeur de l'arbre"); plt.ylabel("Précision"); plt.legend()

### Hack Time

- À partir de quelle profondeur l'overfitting commence-t-il à se voir ?
- Avec `max_depth=None`, l'arbre est-il plus performant qu'avec une profondeur modérée ?
- Quelle profondeur maximise la **performance en test** ? (C'est ce qu'on appelle le *sweet spot*.)

## 2.5 Métriques : accuracy, precision, recall

Jusqu'ici on a regardé `score()`, qui retourne par défaut l'**accuracy** : la proportion de prédictions correctes. C'est intuitif, mais ça ment quand les classes sont déséquilibrées.

Cas typique ici : `y_pass_bar` vaut 1 (réussite) dans ~90 % des cas. Un modèle bidon qui dit **« tout le monde passe »** obtient déjà **90 % d'accuracy** sans rien apprendre. Pratique pour bluffer en réunion, inutile pour décider d'admettre un étudiant.

Trois métriques classiques pour la classification binaire, calculées à partir de la matrice de confusion :

|  | Prédit 1 (passe) | Prédit 0 (rate) |
|---|---|---|
| **Réel 1 (passe)** | TP (vrai positif) | FN (faux négatif) |
| **Réel 0 (rate)**  | FP (faux positif) | TN (vrai négatif) |

- **Accuracy** = (TP + TN) / total. « Sur l'ensemble, combien de prédictions sont bonnes ? »
- **Precision** = TP / (TP + FP). « Quand on prédit que la personne va passer, à quelle fréquence est-on dans le vrai ? »
- **Recall** = TP / (TP + FN). « Parmi celles et ceux qui ont vraiment réussi, combien attrape-t-on ? »

Precision et recall sont en **tension** : on peut maximiser l'un en sacrifiant l'autre (prédire « passe » à tout le monde → recall = 100 %, mais precision = taux de base). Le **F1-score** est leur moyenne harmonique, utile pour les résumer en un seul chiffre.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# clf = LogisticRegression(max_iter=1000)
# clf.fit(X_train, y_train)
# y_pred = clf.predict(X_test)
#
# print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
# print(f"Precision : {precision_score(y_test, y_pred):.3f}")
# print(f"Recall    : {recall_score(y_test, y_pred):.3f}")
# print(f"F1        : {f1_score(y_test, y_pred):.3f}")
# print("Matrice de confusion :")
# print(confusion_matrix(y_test, y_pred))

### Hack Time

- Construisez un modèle bidon qui prédit toujours `1` (réussite). Quelle est son accuracy ? Sa precision ? Son recall ?
- Calculez les **mêmes métriques sur la classe minoritaire** (échec, `y = 0`) en passant `pos_label=0` à `precision_score` et `recall_score`. Quel modèle attrape le mieux les échecs ?
- Bonus fairness : calculez `accuracy_score(y_test[mask], y_pred[mask])` séparément pour `mask = (z_white == 1)` et `mask = (z_white == 0)` (utilisez `df_test["z_white"]`). Le modèle est-il aussi précis dans les deux groupes ? On y revient jeudi.

## 2.6 Et si la cible était continue ? Régression

Jusqu'ici on prédit du **binaire** (passe / rate). Mais souvent on veut prédire une valeur **continue** : un revenu, une note, une durée, un score.

Le dataset Law School fournit aussi `y_zgpa` : la moyenne cumulée en droit, standardisée. On peut donc rejouer le même module en **régression** en changeant trois choses, et **trois choses seulement** :

1. **La cible** : `y_zgpa` (continu) au lieu de `y_pass_bar` (binaire). On charge les fichiers `_for_regression_*.csv`.
2. **Le modèle** : `LinearRegression` au lieu de `LogisticRegression`. (Le mot « régression » dans « logistique » est un piège : la régression logistique fait de la *classification*. Personne n'a réclamé la logique, c'est juste comme ça.)
3. **La métrique** : on ne peut plus parler de pourcentage de prédictions correctes — la prédiction est rarement *exactement* la bonne valeur. On mesure l'**erreur moyenne** :
   - **MAE** (Mean Absolute Error) : moyenne des `|y_pred - y_vrai|`. S'interprète dans l'unité de la cible.
   - **RMSE** (Root Mean Squared Error) : racine de la moyenne des erreurs au carré. Pénalise davantage les grosses erreurs.
   - **R²** : part de la variance de `y` expliquée par le modèle. 1 = parfait, 0 = aussi bon qu'une constante, négatif = pire qu'une constante.

Le reste (features, train/test, overfitting) est **identique** : c'est le même framework.

In [ ]:
# 1. Charger les CSV régression (mêmes individus, même X, autre y)
df_train_reg = pd.read_csv(f"{URL}/law_school_for_regression_train.csv")
df_test_reg = pd.read_csv(f"{URL}/law_school_for_regression_test.csv")

X_train_reg = df_train_reg[FEATURES]
y_train_reg = df_train_reg["y_zgpa"]
X_test_reg = df_test_reg[FEATURES]
y_test_reg = df_test_reg["y_zgpa"]

print(f"y_zgpa train : moyenne {y_train_reg.mean():.3f}, std {y_train_reg.std():.3f}")
print(f"y_zgpa test  : moyenne {y_test_reg.mean():.3f}, std {y_test_reg.std():.3f}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 2. Le même geste qu'en classification, juste un autre modèle
# reg = LinearRegression()
# reg.fit(X_train_reg, y_train_reg)
# y_pred_reg = reg.predict(X_test_reg)
#
# 3. Et trois métriques continues au lieu de accuracy/precision/recall
# mae = mean_absolute_error(y_test_reg, y_pred_reg)
# rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
# r2 = r2_score(y_test_reg, y_pred_reg)
#
# print(f"MAE  : {mae:.3f}")
# print(f"RMSE : {rmse:.3f}")
# print(f"R²   : {r2:.3f}")

### Hack Time

- Quel pourcentage de la variance de `y_zgpa` le modèle linéaire explique-t-il (R²) ? À votre avis, c'est beaucoup ou peu pour des données sociales ?
- Faites un scatter plot `y_test_reg` vs `y_pred_reg`. Si le modèle était parfait, tous les points seraient sur la première bissectrice. Que voit-on en pratique ?
- Réessayez avec `DecisionTreeRegressor(max_depth=10)` à la place de `LinearRegression`. Mêmes 3 métriques. Lequel gagne ?

## Récap module 2

Six concepts qui structurent **tout** le machine learning : features · nettoyage · split · overfitting · métriques · changement de tâche (classification ↔ régression).

L'idée à retenir : le framework est **le même** quel que soit le problème. On change la cible, le modèle, la métrique. Le reste (features, split, garde-fou anti-overfitting) ne bouge pas.

→ Direction le **module 3** : on retourne au texte pour comprendre comment il devient des **tokens**, le carburant des LLMs.